# Benchmark des méthodes de quantization — YOLO26n (détection de chute)

Compare **FP32, FP16, INT8 dynamique, INT8 statique (MinMax / Percentile), QAT et TFLite** sur 4 axes : précision (mAP50, mAP50-95, précision, rappel), taille, latence CPU et temps de préparation.

**Budget temps : ~25 min sur Colab (GPU T4)** — 15–20 min si on ramène `TRAIN_MIN` à 5 et `N_TRAIN` à 1500, au prix d'un FP32 qui ne converge pas (essai précédent : mAP50-95 ≈ 0,006)

| Étape | Durée visée |
|---|---|
| Installation + sous-ensemble du dataset (4000 + 400 images) | ~3–4 min |
| Entraînement FP32 (`TRAIN_MIN`) | 10 min |
| Entraînement QAT (`QAT_MIN`) | 5 min |
| Exports + quantizations + évaluations | ~5–7 min |
| **Total** | **~25 min** (au-delà des 15–20 min : le compromis volontaire pour que le FP32 converge) |

Les durées d'entraînement sont fixées par le paramètre `time=` d'Ultralytics (le nombre d'epochs s'adapte). Pour raccourcir ou allonger, modifiez les constantes de la cellule suivante.

> La latence mesurée est celle du **CPU de Colab** (ONNX Runtime) : elle est indicative. La référence reste la mesure sur le téléphone cible.

In [ ]:
import time, os, json, shutil, random
from pathlib import Path

# ── Paramètres du benchmark (ajuster pour tenir 15-20 min) ───
TRAIN_MIN = 10     # minutes d'entraînement FP32
QAT_MIN   = 5      # minutes d'entraînement QAT (fine-tuning depuis le FP32)
N_TRAIN   = 4000   # images d'entraînement téléchargées
N_VAL     = 400    # images de validation (identiques pour tous les modèles)
N_CALIB   = 100    # images de calibration pour le PTQ statique
IMGSZ     = 640
SEED      = 0
# Réglages d'entraînement adaptés à un budget de quelques epochs seulement :
# warmup court (le défaut, 3 epochs, occuperait presque tout l'entraînement), cache RAM, validation finale seule.
FAST = dict(warmup_epochs=0.5, close_mosaic=1, cache='ram', val=False, patience=100,
            optimizer='AdamW', cos_lr=True, plots=False, workers=4, verbose=False, exist_ok=True)
NDJSON_PATH = 'data/merged-fall-dataset-yolo.ndjson'   # sinon, upload demandé

T0 = time.time()
STAGES = {}          # nom de l'étape -> durée (s)

class stage:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); print(f'▶ {self.name}')
    def __exit__(self, *a):
        STAGES[self.name] = time.time() - self.t
        print(f'  ⏱ {self.name}: {STAGES[self.name]:.0f}s (total {time.time()-T0:.0f}s)')

In [ ]:
with stage('Installation'):
    !pip install -q ultralytics onnx onnxruntime onnxslim onnxconverter-common "nvidia-modelopt[torch]" pandas matplotlib seaborn
import torch
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU (les entraînements seront très limités)')

In [ ]:
# ── Sous-ensemble du dataset (NDJSON -> format YOLO) ─────────
import requests, requests.adapters
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

with stage('Dataset'):
    if not os.path.exists(NDJSON_PATH):
        from google.colab import files
        NDJSON_PATH = list(files.upload().keys())[0]
    rows = [json.loads(l) for l in open(NDJSON_PATH, encoding='utf8')]
    meta, images = rows[0], [r for r in rows[1:] if r.get('type') == 'image']
    names = {int(k): v for k, v in meta['class_names'].items()}
    rng = random.Random(SEED)
    pick = {}
    for sp, n in [('train', N_TRAIN), ('val', N_VAL)]:
        pool = [r for r in images if r['split'] == sp]
        rng.shuffle(pool)
        pick[sp] = pool[:n]

    ROOT = Path('bench_ds')
    shutil.rmtree(ROOT, ignore_errors=True)
    session = requests.Session()
    session.mount('https://', requests.adapters.HTTPAdapter(pool_connections=32, pool_maxsize=32))

    def fetch(args):
        sp, r = args
        img_p = ROOT / 'images' / sp / r['file']
        lab_p = ROOT / 'labels' / sp / (Path(r['file']).stem + '.txt')
        try:
            img_p.write_bytes(session.get(r['url'], timeout=30).content)
            Image.open(img_p).verify()          # rejette les pages d'erreur HTML (URL signée expirée)
        except Exception:
            img_p.unlink(missing_ok=True)
            return False
        boxes = r.get('annotations', {}).get('boxes', [])
        lab_p.write_text('\n'.join(f'{int(b[0])} ' + ' '.join(map(str, b[1:5])) for b in boxes))
        return True

    for sp in pick:
        (ROOT / 'images' / sp).mkdir(parents=True, exist_ok=True)
        (ROOT / 'labels' / sp).mkdir(parents=True, exist_ok=True)
    jobs = [(sp, r) for sp, lst in pick.items() for r in lst]
    with ThreadPoolExecutor(32) as ex:
        ok = list(ex.map(fetch, jobs))
    print(f'{sum(ok)}/{len(jobs)} images téléchargées')
    assert sum(ok) > 0.9 * len(jobs), "Trop d'échecs : régénérez le NDJSON (URLs signées expirées)."

    DATA_YAML = str(ROOT.resolve() / 'data.yaml')
    Path(DATA_YAML).write_text(
        f"path: {ROOT.resolve()}\ntrain: images/train\nval: images/val\nnames:\n" +
        ''.join(f'  {k}: {v}\n' for k, v in names.items()))

## 1. Entraînement FP32 (référence)

In [ ]:
from ultralytics import YOLO
import pandas as pd_

with stage('Entraînement FP32'):
    m = YOLO('yolo26n.pt')
    res = m.train(data=DATA_YAML, time=TRAIN_MIN / 60, epochs=100, imgsz=IMGSZ, batch=32,
                  lr0=0.002, device=DEVICE, name='bench_fp32', **FAST)
    n_ep = len(pd_.read_csv(Path(res.save_dir) / 'results.csv'))
    print(f'{n_ep} epochs réalisées en {TRAIN_MIN} min')
    FP32_PT = Path(res.save_dir) / 'weights' / 'best.pt'
print('FP32 :', FP32_PT)

## 2. Entraînement QAT (fake-quantization INT8)

Les nœuds de fausse quantization (NVIDIA ModelOpt) sont insérés **dans** le modèle FP32 puis réentraînés pendant `QAT_MIN` minutes, afin que les poids s'adaptent à la précision INT8. Si ModelOpt est indisponible ou incompatible, l'étape est notée « échec » dans le tableau au lieu d'interrompre le benchmark.

In [ ]:
QAT_PT, QAT_ERR = None, None
with stage('Entraînement QAT'):
    try:
        import modelopt.torch.quantization as mtq
        from ultralytics.utils.torch_utils import ModelEMA
        import cv2, numpy as np
        calib = sorted((ROOT / 'images' / 'train').glob('*'))[:N_CALIB]

        def insert_quantizers(trainer):
            net = trainer.model
            def loop(model):
                model.eval()
                dev = next(model.parameters()).device
                for p in calib:
                    im = cv2.resize(cv2.imread(str(p)), (IMGSZ, IMGSZ))[:, :, ::-1].transpose(2, 0, 1)[None]
                    with torch.no_grad():
                        model(torch.from_numpy(np.ascontiguousarray(im)).float().to(dev) / 255)
            mtq.quantize(net, mtq.INT8_DEFAULT_CFG, loop)
            trainer.ema = ModelEMA(net)          # l'EMA doit voir les quantizers
            net.train()

        qat = YOLO(str(FP32_PT))
        qat.add_callback('on_pretrain_routine_end', insert_quantizers)
        res = qat.train(data=DATA_YAML, time=QAT_MIN / 60, epochs=100, imgsz=IMGSZ, batch=16,
                        lr0=0.0003, device=DEVICE, name='bench_qat', amp=False, **FAST)
        QAT_PT = Path(res.save_dir) / 'weights' / 'best.pt'
    except Exception as e:
        QAT_ERR = f'{type(e).__name__}: {str(e)[:200]}'
        print('⚠️ QAT indisponible :', QAT_ERR)

## 3. Exports et quantizations post-training

| Variante | Méthode |
|---|---|
| FP32 (ONNX) | export direct |
| FP16 (ONNX) | conversion en float16 (Resize/Upsample laissés en FP32) |
| INT8 dynamique | poids INT8, activations quantifiées à la volée |
| INT8 statique MinMax / Percentile | QDQ par canal, calibration sur `N_CALIB` images, **tête de détection exclue** (restée en FP32 : quantifiée, elle faisait tomber le mAP à 0 lors du premier test) |
| TFLite FP32 / INT8 | export Ultralytics, format utilisé par l'application Android |

In [ ]:
import numpy as np, cv2, onnx, onnxruntime as ort
from onnxruntime.quantization import (quantize_dynamic, quantize_static, CalibrationDataReader,
                                      QuantType, QuantFormat, CalibrationMethod)
from onnxruntime.quantization.shape_inference import quant_pre_process
from onnxconverter_common import float16

ART = Path('bench_models'); ART.mkdir(exist_ok=True)
VARIANTS = []          # dict(label, method, path, error, prep_s)

def add(label, method, path=None, error=None, prep_s=0.0):
    VARIANTS.append(dict(label=label, method=method, path=path, error=error, prep_s=prep_s))

with stage('Export ONNX FP32'):
    t = time.time()
    onnx_fp32 = ART / 'fp32.onnx'
    shutil.copy(YOLO(str(FP32_PT)).export(format='onnx', imgsz=IMGSZ, simplify=True, opset=17), onnx_fp32)
    add('FP32 (ONNX)', 'FP32', onnx_fp32, prep_s=time.time() - t)
    add('FP32 (PyTorch)', 'FP32', FP32_PT)
    pre = ART / 'fp32_pre.onnx'
    quant_pre_process(str(onnx_fp32), str(pre))
    in_name = ort.InferenceSession(str(onnx_fp32), providers=['CPUExecutionProvider']).get_inputs()[0].name

with stage('FP16'):
    t = time.time()
    try:
        # Resize/Upsample bloqués : sinon erreur de type au chargement dans ONNX Runtime
        m16 = float16.convert_float_to_float16(onnx.load(str(onnx_fp32)), keep_io_types=True,
                                               op_block_list=['Resize', 'Upsample'])
        p = ART / 'fp16.onnx'; onnx.save(m16, str(p))
        ort.InferenceSession(str(p), providers=['CPUExecutionProvider'])     # vérifie le chargement
        add('FP16 (ONNX)', 'PTQ FP16', p, prep_s=time.time() - t)
    except Exception as e:
        add('FP16 (ONNX)', 'PTQ FP16', error=f'{type(e).__name__}: {str(e)[:150]}')

with stage('INT8 dynamique'):
    t = time.time()
    p = ART / 'int8_dynamic.onnx'
    quantize_dynamic(str(pre), str(p), weight_type=QuantType.QInt8)
    add('INT8 dynamique (ONNX)', 'PTQ dynamique', p, prep_s=time.time() - t)

class Calib(CalibrationDataReader):
    def __init__(self):
        self.it = iter(sorted((ROOT / 'images' / 'train').glob('*'))[:N_CALIB])
    def get_next(self):
        for p in self.it:
            im = cv2.imread(str(p))
            if im is None: continue
            im = cv2.resize(im, (IMGSZ, IMGSZ))[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255
            return {in_name: np.ascontiguousarray(im)}
        return None

# La tête de détection est le dernier module du modèle : ses nœuds restent en FP32.
head_idx = len(YOLO(str(FP32_PT)).model.model) - 1
head_nodes = [n.name for n in onnx.load(str(pre)).graph.node if f'/model.{head_idx}/' in n.name]
print(f'{len(head_nodes)} nœuds de la tête (model.{head_idx}) exclus de la quantization')

for label, cm, fname in [('INT8 statique MinMax (ONNX)', CalibrationMethod.MinMax, 'int8_static_minmax.onnx'),
                         ('INT8 statique Percentile (ONNX)', CalibrationMethod.Percentile, 'int8_static_percentile.onnx')]:
    with stage(label):
        t = time.time()
        p = ART / fname
        try:
            quantize_static(str(pre), str(p), Calib(), quant_format=QuantFormat.QDQ,
                            activation_type=QuantType.QInt8, weight_type=QuantType.QInt8,
                            per_channel=True, nodes_to_exclude=head_nodes, calibrate_method=cm)
            add(label, 'PTQ statique', p, prep_s=time.time() - t)
        except Exception as e:
            add(label, 'PTQ statique', error=f'{type(e).__name__}: {str(e)[:150]}')

if QAT_PT is None:
    add('QAT (PyTorch)', 'QAT', error=QAT_ERR)
else:
    add('QAT (PyTorch)', 'QAT', QAT_PT, prep_s=STAGES.get('Entraînement QAT', 0))

with stage('TFLite'):
    # Le nom du format/option varie selon la version d'Ultralytics : on essaie plusieurs variantes.
    def try_export(label, method, attempts):
        import glob
        t = time.time(); err = None
        for kw in attempts:
            try:
                out = str(YOLO(str(FP32_PT)).export(imgsz=IMGSZ, **kw))
                if not out.endswith('.tflite'):
                    out = sorted(glob.glob(f'{out}/**/*.tflite', recursive=True))[0]
                dst = ART / (label.lower().replace(' ', '_') + '.tflite'); shutil.copy(out, dst)
                add(label, method, dst, prep_s=time.time() - t); return
            except Exception as e:
                err = f'{type(e).__name__}: {str(e)[:150]}'
        add(label, method, error=err)
    try_export('TFLite FP32', 'FP32 TFLite', [dict(format='litert', quantize=None), dict(format='tflite')])
    try_export('TFLite INT8', 'PTQ INT8 TFLite',
               [dict(format='litert', quantize=8, data=DATA_YAML), dict(format='tflite', int8=True, data=DATA_YAML)])
for v in VARIANTS: print(f"{v['label']:<34} {'OK' if v['path'] else v['error']}")

## 4. Évaluation (mêmes images de validation pour tous les modèles)

In [ ]:
import pandas as pd

def latency_ms(path, runs=30):
    s = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    n = s.get_inputs()[0].name
    x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)
    for _ in range(3): s.run(None, {n: x})
    t = time.perf_counter()
    for _ in range(runs): s.run(None, {n: x})
    return (time.perf_counter() - t) / runs * 1000

rows = []
with stage('Évaluation'):
    for v in VARIANTS:
        row = dict(model=v['label'], method=v['method'], path=str(v['path']) if v['path'] else None,
                   prep_s=round(v['prep_s'], 1), error=v['error'])
        if v['path']:
            try:
                p = Path(v['path'])
                row['size_mb'] = round(p.stat().st_size / 1e6, 2)
                is_pt = p.suffix == '.pt'
                r = YOLO(str(p), task='detect').val(data=DATA_YAML, imgsz=IMGSZ, batch=16 if is_pt else 1,
                                                    device=DEVICE if is_pt else 'cpu', verbose=False, plots=False)
                b = r.box
                row.update({'mAP50': round(b.map50, 4), 'mAP50-95': round(b.map, 4),
                            'precision': round(b.mp, 4), 'recall': round(b.mr, 4)})
                if p.suffix == '.onnx': row['latency_ms'] = round(latency_ms(p), 1)
            except Exception as e:
                row['error'] = f'{type(e).__name__}: {str(e)[:150]}'
        rows.append(row)
df = pd.DataFrame(rows)
for c in ['mAP50', 'mAP50-95', 'precision', 'recall', 'size_mb', 'latency_ms']:
    if c not in df: df[c] = np.nan
base = df.loc[df.model == 'FP32 (ONNX)'].iloc[0]
df['Δ mAP50-95'] = (df['mAP50-95'] - base['mAP50-95']).round(4)
df['taille / FP32'] = (df['size_mb'] / base['size_mb']).round(2)
df['accél. vs FP32'] = (base['latency_ms'] / df['latency_ms']).round(2)

## 5. Tableaux comparatifs

In [ ]:
cols = ['model', 'method', 'size_mb', 'mAP50', 'mAP50-95', 'precision', 'recall', 'latency_ms',
        'Δ mAP50-95', 'taille / FP32', 'accél. vs FP32', 'prep_s']
ok = df[df['mAP50-95'].notna()]
styled = (ok[cols].set_index('model').style
          .background_gradient(subset=['mAP50', 'mAP50-95', 'precision', 'recall'], cmap='Greens')
          .background_gradient(subset=['size_mb', 'latency_ms'], cmap='Reds')
          .format(precision=3, na_rep='—'))
display(styled)
failed = df[df['mAP50-95'].isna()][['model', 'error']]
if len(failed):
    print('Variantes non évaluées :'); display(failed.set_index('model'))

## 6. Graphiques

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
d = ok.reset_index(drop=True)
pal = dict(zip(d.model, sns.color_palette('tab10', len(d))))

fig, ax = plt.subplots(2, 3, figsize=(21, 11))
fig.suptitle('Benchmark de quantization — YOLO26n fall detection', fontsize=16, fontweight='bold')
for a, (col, title, better) in zip(ax.flat, [
        ('mAP50-95', 'mAP50-95', '↑'), ('mAP50', 'mAP50', '↑'), ('recall', 'Rappel (chute manquée = pire erreur)', '↑'),
        ('size_mb', 'Taille (Mo)', '↓'), ('latency_ms', 'Latence CPU (ms, batch 1)', '↓'), ('precision', 'Précision', '↑')]):
    s = d.dropna(subset=[col]).sort_values(col, ascending=(better == '↓'))
    bars = a.barh(s.model, s[col], color=[pal[m] for m in s.model])
    a.set_title(f'{title} {better}', fontweight='bold'); a.invert_yaxis()
    for b_, val in zip(bars, s[col]): a.text(b_.get_width(), b_.get_y() + b_.get_height() / 2, f' {val}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('benchmark_bars.png', dpi=130); plt.show()

In [ ]:
# Compromis précision / taille / latence
fig, ax = plt.subplots(1, 2, figsize=(18, 6))
for a, xcol, xl in [(ax[0], 'size_mb', 'Taille (Mo)'), (ax[1], 'latency_ms', 'Latence CPU (ms)')]:
    s = d.dropna(subset=[xcol])
    for _, r in s.iterrows():
        a.scatter(r[xcol], r['mAP50-95'], s=160, color=pal[r.model], edgecolor='k', zorder=3)
        a.annotate(r.model, (r[xcol], r['mAP50-95']), textcoords='offset points', xytext=(6, 6), fontsize=9)
    a.set_xlabel(xl + '  (← mieux)'); a.set_ylabel('mAP50-95  (↑ mieux)')
    a.set_title(f'mAP50-95 vs {xl}', fontweight='bold')
plt.tight_layout(); plt.savefig('benchmark_tradeoff.png', dpi=130); plt.show()

# Chute de précision par rapport au FP32 ONNX
q = d[d.model != 'FP32 (ONNX)'].sort_values('Δ mAP50-95')
plt.figure(figsize=(12, 0.6 * len(q) + 2))
plt.barh(q.model, q['Δ mAP50-95'], color=['#d62728' if x < -0.02 else '#2ca02c' for x in q['Δ mAP50-95']])
plt.axvline(0, color='k', lw=1); plt.axvline(-0.02, color='gray', ls='--', label='seuil -0,02')
plt.title('Écart de mAP50-95 par rapport au FP32 ONNX', fontweight='bold'); plt.legend()
plt.tight_layout(); plt.savefig('benchmark_delta.png', dpi=130); plt.show()

In [ ]:
# Où est passé le temps ? (cible : ~25 min au total)
total = time.time() - T0
s = pd.Series(STAGES).sort_values()
plt.figure(figsize=(12, 0.5 * len(s) + 2))
plt.barh(s.index, s.values / 60, color='#4A90D9')
plt.xlabel('minutes'); plt.title(f'Durée par étape — total {total / 60:.1f} min', fontweight='bold')
plt.tight_layout(); plt.savefig('benchmark_time.png', dpi=130); plt.show()
print(f'Durée totale : {total / 60:.1f} min', 'OK' if total / 60 <= 25 else '(au-dessus de ~25 min : réduire TRAIN_MIN / QAT_MIN / N_TRAIN)')

## 7. Sélection et sauvegarde du meilleur modèle quantifié

Critère : parmi les variantes **quantifiées** (INT8 / FP16 / QAT / TFLite INT8) dont le mAP50-95 est à moins de 0,02 du FP32, on prend la **plus rapide**, puis la plus petite. Si aucune ne convient, le FP32 reste le modèle à déployer.

In [ ]:
quant = d[~d.model.str.startswith('FP32') & ~d.model.str.startswith('TFLite FP32')]
cand = quant[quant['Δ mAP50-95'] >= -0.02]
if len(cand):
    cand = cand.assign(_lat=cand.latency_ms.fillna(1e9)).sort_values(['_lat', 'size_mb'])
    best = cand.iloc[0]
    print(f"🏆 Meilleur modèle quantifié : {best.model}  (mAP50-95 {best['mAP50-95']}, {best.size_mb} Mo, {best.latency_ms} ms)")
else:
    best = d[d.model == 'FP32 (ONNX)'].iloc[0]
    print('Aucune variante quantifiée ne conserve la précision : le FP32 reste le meilleur choix.')

Path('models').mkdir(exist_ok=True); Path('results').mkdir(exist_ok=True)
src = Path(best.path); dst = Path('models') / f'best_quantized{src.suffix}'
shutil.copy(src, dst)
for f in ['benchmark_bars.png', 'benchmark_tradeoff.png', 'benchmark_delta.png', 'benchmark_time.png']:
    shutil.copy(f, Path('results') / f)
df.to_csv('results/benchmark_results.csv', index=False)
json.dump({'setup': dict(train_min=TRAIN_MIN, qat_min=QAT_MIN, n_train=N_TRAIN, n_val=N_VAL, imgsz=IMGSZ),
           'best_model': best.model, 'best_model_file': str(dst),
           'results': json.loads(df.to_json(orient='records')),
           'stage_seconds': {k: round(v, 1) for k, v in STAGES.items()}},
          open('results/benchmark_results.json', 'w'), indent=2, ensure_ascii=False)
print('💾', dst, '+ results/benchmark_results.(csv|json) + graphiques')
try:
    from google.colab import files
    for f in [str(dst), 'results/benchmark_results.csv', 'results/benchmark_results.json']: files.download(f)
except ImportError:
    pass